In [1]:
import hashlib
from pathlib import Path

In [2]:
import numpy as np
import pandas as pd

In [3]:
# locating the raw data folder 
RAW = Path("..") / "data" / "raw"

In [4]:
# Recorded in docs/SOURCES.md at acquisition, 2026-09-20.
EXPECTED_MD5 = {
    "wfp_food_prices_gha.csv": "1e3aed1c5551f38375b744852baabb4f",
    "wfp_markets_gha.csv": "e234287b980bf36fc165dcfdba914825",
}

In [5]:
Path.cwd()

WindowsPath('C:/Users/hadic/OneDrive/Desktop/Agri-Finance project/notebooks')

In [6]:
for name, expected in EXPECTED_MD5.items():
    actual = hashlib.md5((RAW / name).read_bytes()).hexdigest()
    print(name, actual,"OK" if actual == expected else "CHANGED")
    #using asserts to stop the notebook if the check fails, so the wrong file is not run
    assert actual == expected

print("pandas",pd.__version__)

wfp_food_prices_gha.csv 1e3aed1c5551f38375b744852baabb4f OK
wfp_markets_gha.csv e234287b980bf36fc165dcfdba914825 OK
pandas 3.0.6


In [7]:
# overview of the data

prices = pd.read_csv(RAW / "wfp_food_prices_gha.csv")
markets = pd.read_csv(RAW / "wfp_markets_gha.csv")

print("prices :", prices.shape)
print("markets:", markets.shape)

prices.dtypes 

prices : (37765, 16)
markets: (93, 7)


date                str
admin1              str
admin2              str
market              str
market_id         int64
latitude        float64
longitude       float64
category            str
commodity           str
commodity_id      int64
unit                str
priceflag           str
pricetype           str
currency            str
price           float64
usdprice        float64
dtype: object

In [8]:
# checking for blanks
prices.isna().sum()

date            0
admin1          0
admin2          0
market          0
market_id       0
latitude        0
longitude       0
category        0
commodity       0
commodity_id    0
unit            0
priceflag       0
pricetype       0
currency        0
price           0
usdprice        0
dtype: int64

In [9]:
dates = pd.to_datetime(prices["date"], format = "%Y-%m-%d")

print("first", dates.min().date(), " last:", dates.max().date())
print("days of the month used:", sorted(dates.dt.day.unique().tolist()))

# counting eah year, and how many months appear 
dates.dt.month.groupby(dates.dt.year).nunique()

first 2006-01-15  last: 2023-07-15
days of the month used: [15]


date
2006    12
2007    12
2008    12
2009    12
2010    12
2011    12
2012    12
2013    12
2014    12
2015    12
2016    12
2017    12
2018    12
2019    12
2020    12
2021    12
2022    12
2023     7
Name: date, dtype: int64

In [10]:
attrs = ["market", "admin1", "admin2","latitude","longitude"]

#counting each distinct market_id
# anything above 1 means one id is being used for two different things

conflicts = prices.groupby("market_id")[attrs].nunique()
print("ids with >1 value, per attribute:", (conflicts > 1).sum().to_dict())

# checking for any market name spread across more than one id
print("names with >1 id:", int((prices.groupby("market")["market_id"].nunique() > 1).sum()))

# checking if every id in the price data exist in the markets file
ids_in_prices = prices["market_id"].unique()
print("markets in price file:", len(ids_in_prices), "  in markets file:", len(markets))
print("every price-file id is in the markets file:", bool(np.isin(ids_in_prices, markets["market_id"]).all()))

#checking for markets in both files, do the name, region and coordinates agree
one_row_per_market = prices.drop_duplicates("market_id")[["market_id"] + attrs]
joined = one_row_per_market.merge(markets, on="market_id", suffixes=("", "_mk"))
{c: int((joined[c] != joined[c + "_mk"]).sum()) for c in attrs}

ids with >1 value, per attribute: {'market': 0, 'admin1': 0, 'admin2': 0, 'latitude': 0, 'longitude': 0}
names with >1 id: 0
markets in price file: 44   in markets file: 93
every price-file id is in the markets file: True


{'market': 0, 'admin1': 0, 'admin2': 0, 'latitude': 0, 'longitude': 0}

Section 1: I understood the datatypes for prices,number of nrows = 37,765 and ncols = 16
Section 2: dates has no blank cells, all the 37,765 has 15th as the day, which is not the days a trader was asked a price, but WFP uses that to give value and stamps  
Section 3: this section needs no rule to fix market identity, this was because it was measured. 

Section 4: checked for nrows for retail which I estimated to be <=14445 which was actually 14530 and the number of markets in every year varies. 



In [11]:
#adding parsed date and a year column to returrn a new table; prices remain untouched

p = prices.assign(date=dates, year=dates.dt.year)

per_year = p.groupby("year").agg(
    rows=("price","size"),    #how many rows
    markets=("market_id","nunique"),  #how many different markets
    commodities=("commodity","nunique"),

)

#Add a column per price 
per_year = per_year.join(pd.crosstab(p["year"],p["pricetype"]))

assert per_year["rows"].sum() == len(prices)   # years adding up to the file
assert (per_year["Retail"] + per_year["Wholesale"]).equals(per_year["rows"]) #adding price types to each year

per_year


,rows,markets,commodities,Retail,Wholesale
year,,,,,
2006,382,6,7,0,382
2007,449,6,7,0,449
2008,951,14,8,0,951
2009,1047,15,8,0,1047
2010,1033,15,9,0,1033
2011,1124,15,9,0,1124
2012,1101,15,9,0,1101
2013,1101,15,9,0,1101
2014,1103,43,9,0,1103


In [12]:
#priceflag
pd.crosstab(p["year"], p["priceflag"])

priceflag,actual,"actual,aggregate",aggregate
year,,,
2006,382,0,0
2007,449,0,0
2008,951,0,0
2009,1047,0,0
2010,1033,0,0
2011,1124,0,0
2012,1101,0,0
2013,1101,0,0
2014,1103,0,0


findings: 
    No exact duplicates and no duplicates on the key, 
    so each row is one date + market + commodity + unit + price type. 
        41 cases have the same market, date and commodity priced in two different units (40 yam, 1 rice) 
        → Phase 3 must decide which price counts.

In [13]:
print("exact duplicates rows:", int(p.duplicated().sum()))

key = ["date", "market_id", "commodity_id", "unit", "pricetype"]
print("duplicates on the key:", int(p.duplicated(key).sum()))

# Same test without unit. keep=False marks BOTH rows of each pair, so we can look at them.
key_no_unit = ["date", "market_id", "commodity_id", "pricetype"]
two_units = p[p.duplicated(key_no_unit, keep=False)]
print("rows where one market/date/commodity appears in two units:", len(two_units))
two_units.groupby(["commodity", "unit"]).size()

exact duplicates rows: 0
duplicates on the key: 0
rows where one market/date/commodity appears in two units: 82


commodity     unit      
Rice (local)  100 KG         1
              50 KG          1
Yam           100 Tubers    40
              250 KG        40
dtype: int64

In [14]:
# Number of rows per unit, seperating for retail and wholesale 
p.groupby(["pricetype","unit"]).size()

pricetype  unit      
Retail     KG            14530
Wholesale  10 KG           867
           100 KG         3161
           100 Tubers     1713
           109 KG         2176
           16 KG           233
           20 KG           688
           250 KG         1140
           27 KG           370
           30 pcs          480
           50 KG          4009
           52 KG           829
           68 KG           490
           73 KG           321
           84 KG           297
           91 KG          2205
           93 KG          1503
           Bunch          2753
dtype: int64

In [15]:
# grouping for the six crops 
staples = ["Maize", "Millet", "Sorghum", "Rice (local)", "Cassava", "Yam"]
s = p[p["commodity"].isin(staples)]

# Rows per commodity + unit, with a column each for Retail and Wholesale.
pd.crosstab([s["commodity"], s["unit"]], s["pricetype"])

pricetype                Retail  Wholesale
commodity    unit                         
Cassava      91 KG            0       2205
             KG             645          0
Maize        100 KG           0       2579
             KG             778          0
Millet       93 KG            0       1503
             KG             772          0
Rice (local) 100 KG           0         34
             50 KG            0       2162
             KG             667          0
Sorghum      109 KG           0       1463
             KG             658          0
Yam          100 Tubers       0       1713
             250 KG           0        652
             KG             693          0

Findings
Section 7, redenomination: Checked whether 2006–June 2007 prices were in old or new cedis. The implied exchange rate is 0.92 cedis per dollar from 2006 (old cedis would give ~9,200), and maize moves smoothly through July 2007 (18.3 → 20.9 → 22.0). The file is already in new cedis → Phase 3 does not convert; the cleaning log records it as checked.

In [16]:
# determining the exchange rate f

# price(cedis) / usdprice = cedis/dollar for each row
p["implied_rate"] = p["price"] / p["usdprice"]


# determining the median rate across all rows 
monthly_rate = p.groupby("date")["implied_rate"].median()
monthly_rate.loc["2006-01":"2008-06"].round(3)


date
2006-01-15    0.920
2006-02-15    0.920
2006-03-15    0.920
2006-04-15    0.920
2006-05-15    0.920
2006-06-15    0.920
2006-07-15    0.920
2006-08-15    0.920
2006-09-15    0.920
2006-10-15    0.920
2006-11-15    0.920
2006-12-15    0.920
2007-01-15    0.920
2007-02-15    0.920
2007-03-15    0.920
2007-04-15    0.920
2007-05-15    0.920
2007-06-15    0.920
2007-07-15    0.930
2007-08-15    0.930
2007-09-15    0.950
2007-10-15    0.940
2007-11-15    0.960
2007-12-15    0.950
2008-01-15    0.950
2008-02-15    0.960
2008-03-15    0.960
2008-04-15    0.990
2008-05-15    0.970
2008-06-15    1.073
Name: implied_rate, dtype: float64

In [17]:
# checking if the maize price jumped around July,2007
maize_bag = p[(p["commodity"] == "Maize") & (p["unit"] == "100 KG")]
maize_bag.groupby("date")["price"].median().loc["2007-01":"2007-12"].round(1)

date
2007-01-15    17.5
2007-02-15    18.5
2007-03-15    20.4
2007-04-15    20.0
2007-05-15    20.3
2007-06-15    18.3
2007-07-15    20.9
2007-08-15    22.0
2007-09-15    23.6
2007-10-15    23.5
2007-11-15    27.7
2007-12-15    25.4
Name: price, dtype: float64

Findings 
Section 8, usdprice: Compared each row's exchange rate with its month's median. 92 rows disagree, 82 by exactly 10 times and 10 by exactly 100 times, because usdprice lost trailing zeros (50 → 5.0, 100 → 1.0). price is correct in those rows → don't use usdprice as published; if dollar prices are needed, recalculate them from price and a sourced exchange rate.

In [18]:
# Each row's rate divided by its month's median rate. 1.0 = exactly in line with the month.
# transform("median") puts the month's median beside EVERY row, so each row can be compared.
p["rate_vs_month"] = p["implied_rate"] / p.groupby("date")["implied_rate"].transform("median")

# Keep rows more than 5% away from their month's rate.
off = p[(p["rate_vs_month"] > 1.05) | (p["rate_vs_month"] < 0.95)]
print("rows whose rate disagrees with their month:", len(off))

# How far off, rounded to a whole number, and how many rows at each distance.
print(off["rate_vs_month"].round(0).value_counts())

off[["date", "market", "commodity", "unit", "price", "usdprice"]].head(8)

rows whose rate disagrees with their month: 92
rate_vs_month
10.0     82
100.0    10
Name: count, dtype: int64


,date,market,commodity,unit,price,usdprice
500,2007-04-15,Techiman,Rice (local),50 KG,46.0,5.0
644,2007-08-15,Kumasi,Maize,100 KG,27.9,3.0
649,2007-08-15,Kumasi,Yam,250 KG,93.0,1.0
762,2007-11-15,Techiman,Rice (imported),50 KG,48.0,5.0
804,2007-12-15,Techiman,Sorghum,109 KG,28.5,3.0
808,2007-12-15,Accra,Millet,93 KG,47.5,5.0
852,2008-01-15,Techiman,Sorghum,109 KG,28.5,3.0
896,2008-01-15,Wa,Rice (imported),50 KG,47.5,5.0


Line 

Checked for seasonal patterns per market comparing to markets and regions, found that maize has about 15 markets ay ear from 2009. Millet and sorghum have had only 6 till 2018 and a jump to 14-17 from 2029. this could affect seasonality claim for Upper west in 2018-2020 rests on alomost no data on Wa.

In [19]:
A_CROPS = ["Maize","Millet","Sorghum","Rice (local)","Cassava"]
w = p[(p["pricetype"] == "Wholesale") & p["commodity"].isin(A_CROPS)]

# Fro each crop and year: how many diffrent markets reported wholesale price
w.groupby(["commodity", "year"])["market_id"].nunique().unstack(fill_value=0)

year,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
commodity,,,,,,,,,,,,,,,,,,
Cassava,4,4,12,15,15,15,14,14,38,14,15,14,11,14,13,11,11,14
Maize,6,6,14,15,15,15,15,15,43,15,15,15,15,15,16,16,17,18
Millet,6,6,6,6,6,6,6,6,6,6,6,6,5,14,16,16,17,17
Rice (local),6,6,14,15,15,15,15,15,42,13,14,13,13,13,14,14,15,17
Sorghum,6,6,6,6,6,6,6,6,6,6,6,6,5,14,16,16,17,17


In [20]:
CROP = "Millet"   # change this to look at another crop
LONG_PANEL = ["Wa", "Bolga", "Tamale", "Accra", "Tema", "Kumasi", "Obuasi", "Ejura",
              "Techiman", "Sunyani", "Cape Coast", "Mankessim", "Sekondi/Takoradi",
              "Koforidua", "Ho"]

one = w[(w["commodity"] == CROP) & w["market"].isin(LONG_PANEL)]
pd.crosstab(one["market"], one["year"])

year,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
market,,,,,,,,,,,,,,,,,,
Accra,11,11,12,12,12,10,12,12,5,10,10,11,12,12,11,12,10,6
Bolga,12,12,12,12,12,12,12,12,5,12,12,12,12,12,11,10,9,7
Cape Coast,0,0,0,0,0,0,0,0,0,0,0,0,0,5,12,12,11,7
Ejura,0,0,0,0,0,0,0,0,0,0,0,0,0,5,7,5,1,7
Ho,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
Koforidua,0,0,0,0,0,0,0,0,0,0,0,0,0,5,12,11,2,6
Kumasi,11,11,12,10,11,12,11,11,5,10,10,10,10,12,12,10,7,7
Sekondi/Takoradi,0,0,0,0,0,0,0,0,0,0,0,0,0,5,12,12,10,7
Sunyani,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,11,9,0


Finding 
Checked for suspicious prices in terms of median across 7 months, found that flags before and after 2019 were 98 and 1907 respectively with 'Wholesale': 1014, 'Retail': 991, what phase 3 should do is inspect flags before removing rows and make a decision   

In [21]:
series = ["market_id", "commodity_id", "unit", "pricetype"]
p = p.sort_values(series + ["date"])

# For each price: the median of the 7 months around it, within its own series.
# min_periods=3 means it needs at least 3 prices in that window to calculate.
p["rolling_median"] = (p.groupby(series)["price"]
                         .transform(lambda s: s.rolling(7, center=True, min_periods=3).median()))
p["ratio"] = p["price"] / p["rolling_median"]

flag = p[(p["ratio"] > 2) | (p["ratio"] < 0.5)]
print("flagged:", len(flag), "of", int(p["ratio"].notna().sum()), "rows that could be tested")
print("by price type:", flag["pricetype"].value_counts().to_dict())

fa = flag[(flag["pricetype"] == "Wholesale") & flag["commodity"].isin(A_CROPS)]
print("Project A crops, wholesale:", len(fa), fa["commodity"].value_counts().to_dict())

flagged: 2005 of 37638 rows that could be tested
by price type: {'Wholesale': 1014, 'Retail': 991}
Project A crops, wholesale: 77 {'Cassava': 70, 'Maize': 4, 'Rice (local)': 2, 'Sorghum': 1}


In [22]:
cols = ["date", "market", "commodity", "unit", "pricetype", "price", "rolling_median", "ratio"]
flag.sort_values("ratio", ascending=False)[cols].head(8).round({"price": 2, "rolling_median": 2, "ratio": 2})

,date,market,commodity,unit,pricetype,price,rolling_median,ratio
15794,2020-02-15,Kumasi,Eggplants,27 KG,Wholesale,370.40,2.38,155.63
15796,2020-02-15,Kumasi,Onions,73 KG,Wholesale,196.83,2.71,72.63
15890,2020-02-15,Sunyani,Onions,73 KG,Wholesale,318.94,5.32,59.95
15919,2020-02-15,Techiman,Onions,73 KG,Wholesale,487.33,8.36,58.29
15998,2020-02-15,Koforidua,Onions,73 KG,Wholesale,333.33,6.00,55.56
15888,2020-02-15,Sunyani,Eggplants,27 KG,Wholesale,257.05,4.78,53.78
16089,2020-02-15,Tamale,Soybeans,KG,Retail,28.00,0.59,47.46
15805,2020-02-15,Kumasi,Tomatoes (local),52 KG,Wholesale,751.82,16.48,45.62


In [23]:
# True = 2019 or later, False = before 2019. Each number is a count of rows.
print("flagged:", flag.groupby(flag["year"] >= 2019).size().to_dict())
tested = p[p["ratio"].notna()]
print("tested: ", tested.groupby(tested["year"] >= 2019).size().to_dict())

flagged: {False: 98, True: 1907}
tested:  {False: 11702, True: 25936}


Findings
Section 11, Upper East millet: Checked Phase 1's August–September price low against the November–December harvest. That low came from averaging raw prices across years: 2023, the most expensive year (7.91 cedis/kg), only covers January–July. Compared with each month's local trend, Bolga millet is cheapest January–April (0.898–0.938, 7–10% below trend), dearest in June (1.100), and at trend in August–September → the calendar and prices agree; the Phase 1 mismatch was a measurement problem.




In [24]:
b = p[(p["market"] == "Bolga") & (p["commodity"] == "Millet")].copy()
b["month"] = b["date"].dt.month

#putting every price in cedis/kg: bag prices / 93; kg remains the same
b["per_kg"] = b["price"] / np.where(b["unit"] == "93 KG", 93, 1)

phase1   = b.groupby("month")["per_kg"].mean()                                         # Phase 1: everything together
bag_only = b[b["unit"] == "93 KG"].groupby("month")["per_kg"].mean()                     # wholesale bag only
no_2023  = b[(b["unit"] == "93 KG") & (b["year"] < 2023)].groupby("month")["per_kg"].mean()

pd.DataFrame({"A. Phase 1 method": phase1,
              "B. bag only": bag_only,
              "C. bag only, 2023 removed": no_2023}).round(2).T

month,1,2,3,4,5,6,7,8,9,10,11,12
A. Phase 1 method,2.22,2.18,2.04,2.48,2.37,2.20,2.40,1.64,1.57,1.82,1.89,1.82
B. bag only,1.69,1.65,1.55,1.84,1.81,1.70,1.95,1.40,1.42,1.70,1.80,1.73
"C. bag only, 2023 removed",1.26,1.29,1.16,1.47,1.45,1.33,1.59,1.40,1.42,1.70,1.80,1.73


In [25]:
# average in price in cedis/kg 

bag = b[b["unit"] == "93 KG"]
print("average cedis per kg, by year:", bag.groupby("year")["per_kg"].mean().round(2).to_dict())
print("months present in 2023:", sorted(bag.loc[bag["year"] == 2023, "month"].tolist()))

average cedis per kg, by year: {2006: 0.29, 2007: 0.3, 2008: 0.58, 2009: 0.6, 2010: 0.56, 2011: 0.69, 2012: 1.18, 2013: 1.08, 2014: 1.09, 2015: 1.42, 2016: 1.54, 2017: 2.07, 2018: 2.3, 2019: 1.59, 2020: 2.13, 2021: 3.02, 2022: 5.59, 2023: 7.91}
months present in 2023: [1, 2, 3, 4, 5, 6, 7]


In [26]:
def seasonal_index(market, commodity, unit):
    # One market's wholesale series for one crop, ordered by date.
    s = (p[(p["market"] == market) & (p["commodity"] == commodity) &
           (p["unit"] == unit) & (p["pricetype"] == "Wholesale")]
         .set_index("date")["price"].sort_index())
    # Put every month on the calendar, so a missing month becomes a gap instead of being skipped.
    s = s.reindex(pd.date_range(s.index.min(), s.index.max(), freq="MS") + pd.Timedelta(days=14))
    # The trend at each month: the average of the 12 months around it.
    trend = s.rolling(12).mean().rolling(2).mean().shift(-6)
    # Each month's price as a share of its trend. 0.90 = 10% below trend.
    ratio = (s / trend).dropna()
    return ratio.groupby(ratio.index.month).agg(["median", "count"]).T.round(3)

seasonal_index("Bolga", "Millet", "93 KG")

,1,2,3,4,5,6,7,8,9,10,11,12
median,0.898,0.917,0.938,0.932,1.027,1.1,1.082,1.038,1.009,1.012,1.0,0.945
count,12.000,12.000,12.000,11.000,11.000,11.0,12.000,13.000,13.000,12.000,12.0,12.000
